In [1]:
import os

In [2]:
%pwd

'c:\\Users\\banot\\text-summarizer-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\banot\\text-summarizer-'

In [5]:
import sys
import os
sys.path.append(os.path.abspath("src"))

In [ ]:
from  dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path


In [8]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml,create_directories



In [9]:
import sys
!{sys.executable} -m pip install pyyaml

In [10]:
import yaml
print("working")

working


In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [12]:
import os
import urllib.request as request
import zipfile
from textsummarizer.logging import logger
from textsummarizer.utils.common import get_size

In [25]:
import os
import urllib.request as request
import zipfile
import ssl
from textsummarizer.logging import logger
from textsummarizer.utils.common import get_size

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            # Create SSL context that bypasses certificate verification
            ssl_context = ssl._create_unverified_context()
            
            # Use urlopen instead of urlretrieve to support SSL context
            with request.urlopen(self.config.source_URL, context=ssl_context) as response:
                with open(self.config.local_data_file, 'wb') as out_file:
                    out_file.write(response.read())
            
            logger.info(f"{self.config.local_data_file} download completed")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  

        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [26]:
import yaml
with open ("config/config.yaml") as f:
    data = yaml.safe_load(f)
print(data)
print(type(data))    


{'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}}
<class 'dict'>


In [27]:
config

In [28]:
from pprint import pprint
config = ConfigurationManager()
pprint(config.config)

[2026-06-25 15:18:07,446: INFO: common: YAML file loaded successfully: config\config.yaml]
[2026-06-25 15:18:07,449: INFO: common: YAML file loaded successfully: params.yaml]
[2026-06-25 15:18:07,451: INFO: common: Created directory at: artifacts]
{'artifacts_root': 'artifacts',
 'data_ingestion': {'local_data_file': 'artifacts/data_ingestion/data.zip',
                    'root_dir': 'artifacts/data_ingestion',
                    'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip',
                    'unzip_dir': 'artifacts/data_ingestion'}}


In [29]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
    
    print("data ingestion completed successfully")
    
except Exception as e:
    print("error occurred:")
    raise e

[2026-06-25 15:18:11,044: INFO: common: YAML file loaded successfully: config\config.yaml]
[2026-06-25 15:18:11,046: INFO: common: YAML file loaded successfully: params.yaml]
[2026-06-25 15:18:11,047: INFO: common: Created directory at: artifacts]
[2026-06-25 15:18:11,048: INFO: common: Created directory at: artifacts/data_ingestion]
[2026-06-25 15:23:31,522: INFO: 3462961003: artifacts/data_ingestion/data.zip download completed]
data ingestion completed successfully
